# L1a: Introduction to Data Types

Every value in a computer program has a type. We will connect Julia's primitive, collection, and composite types to representation, valid operations, and engineering-program design.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Inspect a value's representation:__ Determine the type, storage size, and bit pattern of any Julia value using [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof), [the `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D), and [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring). Together these answer what a value is, how much memory it occupies, and exactly which bits are stored.
> * __Choose a collection type:__ Distinguish tuples, arrays, sets, and dictionaries by their order, mutability, and lookup behavior. Selecting a container is a design decision driven by what the data must support, not a matter of taste.
> * __Build composite types:__ Construct immutable and mutable structs that group related fields under one name. The `mutable` keyword decides whether a value's fields can change after it is constructed.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Everything in this notebook uses only Julia's `Base` library, so no external packages are required here. See [the Julia programming language documentation](https://docs.julialang.org/en/v1/) for the functions and types we use below.

___

## Primitive Data Types
Primitive data types are the basic building blocks a language provides. They are _atomic_: they are not composed of other types, and they hold simple values such as numbers, characters, and truth values.

> __Why the type matters:__
>
> A type is not a label the language attaches for bookkeeping. It fixes how many bytes a value occupies, how the bit pattern in those bytes is interpreted, and which operations the compiler will permit. Two values with identical bits can denote entirely different numbers under two different types.

We start with [Integers](https://docs.julialang.org/en/v1/base/numbers/#Core.Int) and [the `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool), then turn to floating-point values and characters.

### Integer and Boolean Types
An __integer__ represents a whole number $x\in\mathbb{Z}$: positive, negative, or zero. Julia stores integers in a _fixed-width_ binary form, typically 32 or 64 bits, which is what the `32` and `64` in `Int32` and `Int64` refer to. A __boolean__ represents a truth value, either `true` or `false`.

We will ask three questions of each: what is its type, what bits are stored, and how much space does it take. Let's bind a whole number to `x::Int64` and start there.

In [ ]:
x = 2 |> Int64; # select a whole number ... -2, -1, 0, 1, 2, ...

Every Julia value carries its own type, and [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) reports it:

In [ ]:
typeof(x) # this returns the type of the argument

The type tells us how the bits are _interpreted_. [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) shows us the bits themselves.

> __Reading a bitstring:__
>
> The result has one character per bit, most significant bit first, so an `Int64` produces a 64-character string. This is the literal content of memory, not a decimal rendering of it. Only primitive types have a bitstring, because only they occupy a single contiguous block of fixed width.

So what is actually stored for `x`?

In [ ]:
bitstring(x) # shows the bit pattern stored in memory

Now the same three questions for a boolean. A variable of type `Bool` ranges over $\mathbb{B} = \left\{\text{true},\text{false}\right\}$, so it carries exactly one bit of information. Let's bind `false` to `flag::Bool`:

In [ ]:
flag = false; # the flag variable can take on values of {true | false}

The pattern is the same as before. First the type:

In [ ]:
typeof(flag)

Then the stored bits:

In [ ]:
bitstring(flag) # this should be 8 bits wide

A `Bool` carries a single bit of information, but memory is addressed in __bytes__. Let's use [the `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) to see how much space our `flag` variable actually takes up:

In [ ]:
sizeof(flag) # number of bytes used to store the Bool (not sizeof(x) -- x is an Int64!)

___

### Floating point types
Floating-point types model real numbers using three components according to [the IEEE 754 standard](https://en.wikipedia.org/wiki/IEEE_754): a sign bit, an exponent (the scale), and a significand. Only the _fractional part_ of the significand is stored in memory; its leading digit is implicit. We take a floating-point number apart bit by bit in `L1d`, where that implicit leading digit turns out to matter.

> __Julia versus Python floating point numbers__: Julia provides three standard IEEE-754 floating-point types that trade off precision for storage: `Float16` (half-precision), `Float32` (single-precision), and `Float64` (double-precision). Python's built-in `float` type is always 64-bit double-precision.

Let's look at a couple of examples. First, here's a 64-bit number (Julia's default):

In [ ]:
let
    x = 54.13; # default: in Julia, the default floating point number is 64-bit.
    bitstring(x)
end

The same numerical value stored in 32-bits has a different memory layout:

In [ ]:
let
    x = 54.13 |> Float32 # cast to Float32 (single precision), not Float64
    bitstring(x) # gives a string with the bit pattern
end

Fewer bits means less storage — and, as we will see in `L1d`, less precision. `Float16` halves the width again:

In [ ]:
let
    x = 54.13 |> Float16 # cast to Float16 (half precision), not Float64
    sizeof(x) # returns number of bytes used to store x
end

___

### Character Types
Text on computers is composed of characters, and each character is associated with a unique integer called its __code point__. Traditional systems used [ASCII](https://en.wikipedia.org/wiki/ASCII) with one byte per character, while modern systems use [Unicode encodings like UTF-8 or UTF-16](https://en.wikipedia.org/wiki/Unicode) to represent a much wider range of characters.

> __What a `Char` actually is:__
>
> It is tempting to say characters "are" integers, but in Julia `Char <: Integer` is `false`. A [Char](https://docs.julialang.org/en/v1/base/strings/#Core.Char) is its own primitive type that _converts to and from_ integers.
>
> Character encodings define the mapping between textual symbols and numeric code points, enabling text to be stored and transmitted as bytes. Julia's `Char` is a 4-byte (32-bit) primitive — but the bits it stores are the character's __UTF-8 bytes__, left-aligned in the word, _not_ the code point. `UInt32(c)` converts to the code point; it does not simply reinterpret the bits. We will see the difference below.

Let's explore [the `Char` type in Julia](https://docs.julialang.org/en/v1/manual/unicode-input/) (notice the single quotes):

In [ ]:
c = '🍣' # example Unicode character in Julia. See: https://docs.julialang.org/en/v1/manual/unicode-input/

What is the code point (the unique integer) for the character `c`? We convert it with [the `UInt32(...)` constructor](https://docs.julialang.org/en/v1/base/numbers/#Core.UInt32):

In [ ]:
code = UInt32(c) # extract code point as UInt32 (4 x bytes)

__Stored bits versus code point.__ The callout above claimed a `Char` holds UTF-8 bytes rather than the code point. Let's check that claim directly:

In [ ]:
(codepoint = string(UInt32(c), base = 16, pad = 8), # what UInt32(c) converts to
 stored     = string(reinterpret(UInt32, c), base = 16, pad = 8)) # what is actually in the 4 bytes

_Hmmm, what?_ That's a strange-looking integer! The `code::UInt32` is a [hexadecimal number](https://en.wikipedia.org/wiki/Hexadecimal), i.e., a number written in base 16. The giveaway (which is a convention) is the `0x` prefix. We'll dig into these numbers and examine representations in different bases later.

Can we see the data that each byte contains? Yes! Let's use [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret) and break the 4 bytes into four 1-byte blocks!

In [ ]:
reinterpret(Tuple{UInt8, UInt8, UInt8, UInt8}, code) |> collect

This factors the 32-bit value into four 8-bit (1-byte) values. Notice we list bytes from least significant to most significant (right to left) on little‑endian hosts (this corresponds to how `bitstring` shows bits on such machines).  
> __Endianness:__ This ordering relates to [Endianness](https://en.wikipedia.org/wiki/Endianness), which describes how computers store the bytes of multi-byte values. In little-endian systems (like most x86/x86-64 and ARM machines), the least significant byte comes first in memory, while big-endian systems store the most significant byte first. So when we reinterpret 0x0001F363 as four UInt8s on a little-endian machine, we get: `[0x63,0xF3,0x01,0x00]`  

Any `isbits` value can be split into bytes this way — but that does __not__ make a `Char` a collection. [`isprimitivetype(Char)`](https://docs.julialang.org/en/v1/base/base/#Base.isprimitivetype) is `true`: it has a fixed width and no independently addressable elements. Collection types are genuinely different — they hold a variable number of elements you can index, add to, or remove. Let's look at those next.

___

## Collection Types
A collection type is a composite data structure aggregating multiple values, often of the same or related types, into a single container (e.g., tuples, arrays, sets, and dictionaries). It is not itself a primitive type, and its elements need not be primitives either — they may be other collections or composite types. Let's look at a few examples of collections, starting with one that we have already seen (sort of), namely [Tuples](https://docs.julialang.org/en/v1/manual/functions/#Tuples).

### Tuples
A tuple is an immutable, ordered collection of elements that can hold a fixed number of items, potentially of different types. Once created, their size and contents cannot be changed, making tuples useful for grouping related values without the overhead of a mutable container.

> **Julia tuple memory layout:** Every tuple in Julia is an immutable composite object with a type that encodes its length and element types (e.g., `Tuple{Int64, Float64}`). The memory layout is a contiguous block of fields: if all elements are "isbits" (primitives), the tuple itself is isbits and can be unboxed (often in registers or on the stack). However, a non-isbits element (like a `String`) is stored as a [reference to a heap-allocated object](https://en.wikipedia.org/wiki/Pointer_(computer_programming)); any `isbits` fields beside it still sit inline. Where the tuple itself lives — register, stack, or heap — is the compiler's decision, not something the type alone determines.

Let's explore tuples with a concrete example. Since [Tuple types](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable, they can't be changed once constructed.

The `example_tuple::Tuple{Int64, Float64}` variable below holds two values of different types: an age in years and a measurement. The type itself records both the length and the element types.

In [ ]:
example_tuple = let
    pair = (18,36.6); # populate with data. Notice not the same type for each element
end;

What is the type of the `example_tuple` variable? Let's use [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) to find out.

In [ ]:
typeof(example_tuple)

Tuples are immutable. Let's try to change a value in the `example_tuple::Tuple{Int64, Float64}` variable. This should blow up, because [Tuples in Julia](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable.

> **Try-catch blocks:** The `try-catch` construct allows us to handle errors gracefully instead of crashing the program. 
> Code in the `try` block is executed, and if an error occurs, execution jumps to the `catch` block where we can handle the error (like printing a message) rather than terminating the program.  The program continues executing normally after the `catch` block.

So what happens?

In [ ]:
try
    example_tuple[1] = 6 # this will raise an error because tuples are immutable
catch e
    println("expected error: ", e)
end
println("After the try-catch block, the program continues executing normally.")

What does the bitstring look like for the `example_tuple::Tuple{Int64, Float64}` variable?

In [ ]:
try
    bitstring(example_tuple) # Can't get the bitstring directly; a Tuple is not a primitive type.
catch e
    println("expected error: ", e)
end

However, we can get the elements of `example_tuple` and their bit layouts by [indexing into the Tuple](https://docs.julialang.org/en/v1/base/base/#Core.Tuple). For example, let's look at the second element:

In [ ]:
bitstring(example_tuple[2]) # get the bitstring of the component i

We can see the raw bytes associated with the `example_tuple::Tuple{Int64,Float64}` using [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret). Note: this works because the tuple is composed of `isbits` elements and the total size aligns; the exact byte order and layout you see will reflect host endianness and alignment.

In [ ]:
v = reinterpret(NTuple{16,UInt8}, example_tuple) |> collect # we have 16 8-bit blocks (128 bits total)

In [ ]:
bitstring(v[2])

___

### Arrays
An array is a contiguous, ordered collection of elements of the same type, allowing constant-time access to its elements via integer indices. In most languages, arrays occupy a single block of memory, with element access computed as the base address plus the index times the memory size of each element.

> **Julia vs. Python arrays:** [Julia's `Array{T}` type](https://docs.julialang.org/en/v1/base/arrays/#Core.Array-Tuple%7BNothing,%20Any%7D) is a built-in, statically typed container that is `1-indexed` and stored [in column-major order](https://en.wikipedia.org/wiki/Row-_and_column-major_order). Python's native lists are heterogeneous and zero-indexed, while [NumPy's homogeneous arrays](https://numpy.org/doc/stable/reference/generated/numpy.array.html) are zero-indexed and row-major (implemented in a separate C library rather than the core language).

Arrays in both Julia and Python are mutable, meaning elements can be changed after we populate the array. Let's explore a Julia array:

In [ ]:
a = rand(10) # build a 10-element random array

We access the elements of an array by passing the index of the array in square brackets, e.g., `a[3]` returns the third element in Julia (because it is `1`-based):

In [ ]:
a[3]

Arrays are __mutable__, i.e., we can change them after we build them. For example:

In [ ]:
a[3] = π

In [ ]:
a

Arrays in Julia are `1`-based, unlike C, Python, and Java, which are `0`-based.
> __Note:__ This is a deliberate choice, and Julia is in good company: Fortran, MATLAB, and R — the languages scientific computing grew up on — are all `1`-based. The practical argument is that indices line up with the mathematics you are transcribing. When you write $\sum_{i=1}^{n}a_{i}$, the loop is `for i ∈ 1:n` and `a[1]` really is $a_{1}$. The cost is real too: most algorithms in the CS literature are written `0`-based, so translating them takes care.

What happens if we try to grab an element that is _outside_ the array?

In [ ]:
try
    a[11] # asking for index 11, but the array has only 10 items
catch e
    println("expected error: ", e)
end

___

### Sets and Dictionaries
A [Set type](https://docs.julialang.org/en/v1/base/collections/#Base.Set) is an unordered collection of unique elements that supports fast membership checks, insertions, and removals. A [Dictionary (or map) is an associative container](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) that stores key–value pairs, allowing lookup, insertion, and deletion of values based on their unique keys.

> **Julia vs. Python collections:** Julia's `Set{T}` and `Dict{K,V}` are parametric containers, meaning every element in a `Set` has the same type `T`, and every key–value pair in a `Dict` has types `K` and `V`. However, the elements can be any type `T`, and the keys `K` and values `V` can also be of any type. In contrast, Python's built-in `set` and `dict` are inherently heterogeneous (each slot holds a generic `object` reference), making them more flexible than their Julia equivalents.

Let's build a few examples of set and dictionary collection types. The `d::Dict{Int64, String}` variable below models the lines of a text file: each key is a line number, and each value is the text on that line.

In [ ]:
d = let

    d = Dict{Int64, String}(); # creates a dictionary that models text in a file.
    d[1] = "This is the first line in a text file";
    d[2] = "This is the second line in a text file";
    d[3] = "This is the last line in a text file";

    d
end

We can access the values stored in a dictionary by passing in the `key` pointing to a `value`, i.e., to get line `2`, we would:

In [ ]:
d[2]

Dictionaries (in general) do __not__ guarantee insertion order. Look at the output above: we inserted key `1` first, but it printed __last__. The iteration order comes from the hash table's internal layout, not from the order you inserted, and it can change across Julia versions or with a different set of keys — so never rely on it. If you need a map that preserves insertion order, consider `OrderedDict` from the `DataStructures.jl` package. Likewise, there is no notion of order in a set.

Consider the `s::Set{Char}` example:

In [ ]:
s = let

    s = Set{Char}(); # empty at this point
    push!(s, 'a'); # add items to the set using `push!`
    push!(s, 'b');
    push!(s, 'c');
    push!(s, 'd');

    s
end

We can't access a particular item in the `s::Set{Char}` set by passing in an index (or key) because these concepts don't apply to sets. Instead, we can use [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) to pop (get) an arbitrary element from a set:

In [ ]:
pop!(s)

All the typical mathematical operations on sets, such as intersection, union, or membership checks, are implemented in most modern programming languages, including Julia; [see the documentation for operations on sets in Julia](https://docs.julialang.org/en/v1/base/collections/#Set-Like-Collections).

___

## Custom composite types
Custom composite types are user-defined data structures that aggregate multiple fields (possibly of different types) under a single name, enabling encapsulation of related data. Think of them as custom containers that you design to hold exactly the data you need for your specific problem.

> **Language differences:** In Julia, these are declared [using the struct keyword](https://docs.julialang.org/en/v1/manual/types/#Composite-Types) with a list of named fields, similar to C. Python uses classes with attributes and methods, which is a more object-oriented approach.

We'll explore this topic in much greater depth later, but for now, let's build some simple examples to illustrate how composite types work in Julia.

In [ ]:
struct MyStudentModel

    # data -
    firstname::String # fields hold the data, they have names and types
    lastname::String
    id::Int64

    MyStudentModel(f,l,id) = new(f,l,id); # constructor
end

The `MyStudentModel(f,l,id) = new(f,l,id)` line is an [inner constructor](https://docs.julialang.org/en/v1/manual/constructors/#man-inner-constructor-methods); [`new`](https://docs.julialang.org/en/v1/manual/constructors/#man-inner-constructor-methods) is the keyword that actually builds the instance and is only available inside the type definition.

Now we can create a `model::MyStudentModel` instance by calling that constructor:

In [ ]:
model = MyStudentModel("Test", "Student", 1234)

We access the data stored in our composite type using dot syntax:

In [ ]:
model.id # returns the value stored in the id field

Here's a key point: because we used the `struct` keyword, our student model is immutable. Once we build it, we cannot change any of the data stored in the model. Let's see what happens when we try:

In [ ]:
try
    model.id = 5678 # we are trying to change an immutable struct.
catch e
    println("expected error: ", e)
end

Sometimes we need to modify our data after creating it. For these cases, we can create [mutable composite types](https://docs.julialang.org/en/v1/manual/types/#Mutable-Composite-Types) by adding the `mutable` keyword when declaring the struct:

In [ ]:
mutable struct MyMutableStudentModel

     # data -
    firstname::String # fields hold the data, they have names and types
    lastname::String
    id::Int64

    MyMutableStudentModel() = new(); # builds an empty model

end

We create mutable composite types the same way as immutable ones, by calling the constructor. However, the empty constructor `new()` produces an instance whose fields are uninitialized, so you must assign every field before reading it.

The `mutable_model::MyMutableStudentModel` variable below is built that way: the `let` block constructs an empty instance, fills in each field, and returns the populated model. Wrapping this in a `let` block keeps the intermediate `model` name out of the global namespace.

In [ ]:
mutable_model = let

    model = MyMutableStudentModel(); # empty: fields are uninitialized
    model.firstname = "Firstname";
    model.lastname = "Lastname";
    model.id = 6789;

    model # return the populated model
end

___

## Lab
In Lab `L1b`, we will make sure everyone's machines are set up properly for the course. This includes installing Julia, setting up Jupyter notebooks, and ensuring all necessary packages are available.

___

## Summary
Every Julia value carries a type that fixes how it is stored in memory and which operations are valid on it.

> __Key Takeaways:__
>
> * **Types constrain representation and operations:** A type determines both the bit pattern a value occupies and what a program is allowed to do with it.
> * **Collection choice follows the problem:** Tuples, arrays, sets, and dictionaries differ in order, mutability, and lookup, so the container you pick depends on what the data must support.
> * **Composite types encode vocabulary:** A `struct` names the fields a problem needs, and the `mutable` keyword decides whether those fields can change after construction.

Choosing a representation comes before computing anything, so these distinctions carry through every program we write for the rest of the course.
___